# 02 — Entity-resolution spike (task 0.2b)

**Question:** can Oracle's Elixir games be joined to Leaguepedia games *mechanically*?
**Method:** fetch 15 LCK games from Leaguepedia, match them to OE via (date ±1, set of 10 picked champions), verify team names.

Pipeline: setup → resume state from disk → Data Dragon map → fetch details (cached) → OE side → match → decision note.
This notebook is restart-proof: every API result is pickled, re-running never refetches what's on disk.

In [1]:
# ── Setup: imports, login, paths ─────────────────────────────────────────
import os, time, pickle
import duckdb
import requests
from dotenv import load_dotenv
import leaguepedia_parser as lp
from leaguepedia_parser.site.leaguepedia import leaguepedia
from mwclient.errors import APIError, MaximumRetriesExceeded, LoginError
from requests.exceptions import ConnectionError as RequestsConnectionError

load_dotenv("../.env")
try:
    leaguepedia.site.client.login(os.getenv("LP_USER"), os.getenv("LP_PASS"))
except LoginError as e:
    if "BotPasswordSessionProvider" in str(e):
        pass          # already logged in this session — re-running the cell is safe
    else:
        raise         # real login failure: bad credentials etc.

CSV = "../data/2025_LoL_esports_match_data_from_OraclesElixir.csv"
SKELETONS = "../data/lck_2025_r12_games.pkl"
CACHE = "../data/lp_details_cache.pkl"


In [2]:
# ── Resume state from disk (never refetch what we already paid for) ──────
if os.path.exists(SKELETONS) and os.path.getsize(SKELETONS) > 0:
    with open(SKELETONS, "rb") as f:
        games_lp = pickle.load(f)
else:
    games_lp = lp.get_games("LCK/2025 Season/Rounds 1-2")   # verified overviewPage
    with open(SKELETONS, "wb") as f:
        pickle.dump(games_lp, f)

cache = pickle.load(open(CACHE, "rb")) if os.path.exists(CACHE) and os.path.getsize(CACHE) > 0 else {}
print(len(games_lp), "skeletons |", len(cache), "detailed in cache")


222 skeletons | 15 detailed in cache


## Findings from exploration (cells removed; results kept)

- **Two-tier API:** `get_games(overviewPage)` → cheap skeletons (champs, bans, `gameInSeries`, patch, start time, winner). `get_game_details(game)` → expensive, fires **2 parallel requests**, fills roles, player identity, `picksBans`.
- **Identity hides in `.sources`:** `game.teams.BLUE.sources.leaguepedia.name` → `'Hanwha Life Esports'`; `player.sources.leaguepedia` → `name='Zeus'`, `irlName`, `country`, `birthday`. Top-level `vars()` doesn't recurse — always check `.sources`.
- **`picksBans` is the full interleaved draft** and matches the fixed tournament format exactly (B-R-B-R-B-R bans / B-RR-BB-R picks / R-B-R-B bans / R-BB-R picks) → OE per-team order + format rules reconstructs the global sequence. **Verified.**
- **Patch scheme differs:** OE `15.06` (double! loses trailing zeros: `15.1`) vs LP `'25.06'` (string). → store patch as VARCHAR, reconcile on the suffix.
- **Champion encoding differs:** OE names vs LP numeric Riot IDs → bridge via Data Dragon.
- **Roles:** OE `top/jng/mid/bot/sup` vs LP `TOP/JGL/MID/BOT/SUP`.
- **Rate limits:** hostile even authenticated; connection drops (`RemoteDisconnected`) as well as polite `ratelimited` errors. Much faster in the morning (EU) than late night. → backoff + incremental cache + schedule bulk fetches.

In [3]:
# ── Data Dragon: champion name → Riot numeric ID (no API key, no limits) ─
versions = requests.get("https://ddragon.leagueoflegends.com/api/versions.json").json()
champs = requests.get(
    f"https://ddragon.leagueoflegends.com/cdn/{versions[0]}/data/en_US/champion.json"
).json()["data"]
name_to_id = {c["name"]: int(c["key"]) for c in champs.values()}
len(name_to_id)


173

In [4]:
# ── Robust fetch: survives rate limits AND connection drops ──────────────
RETRYABLE = (APIError, MaximumRetriesExceeded, RequestsConnectionError)

def fetch_details_safely(game, max_retries=6):
    wait = 60
    for attempt in range(max_retries):
        try:
            return lp.get_game_details(game)
        except RETRYABLE as e:
            print(f"{type(e).__name__}, sleeping {wait}s (attempt {attempt+1})")
            time.sleep(wait)
            wait *= 2
    raise RuntimeError("server refused for a long time — stop and try again later")


In [5]:
# ── Fetch details for the 15-game sample (incremental cache, resumable) ──
def game_key(g):
    picks = frozenset(pb.championId for pb in g.picksBans if not pb.isBan) if g.picksBans \
        else frozenset(p.championId for p in g.teams.BLUE.players) | \
             frozenset(p.championId for p in g.teams.RED.players)
    return (g.start, picks)

for g in games_lp[:15]:
    if game_key(g) in cache:
        continue                              # already fetched in a previous run
    cache[game_key(g)] = fetch_details_safely(g)
    pickle.dump(cache, open(CACHE, "wb"))     # save after EVERY success
    print(f"{len(cache)} cached")
    time.sleep(20)                            # details fire 2 parallel requests — be generous

print("done:", len(cache), "in cache")


done: 15 in cache


In [6]:
# ── OE side: one row per game with date, champion-ID set, team names ─────
oe = duckdb.sql(f'''
    SELECT gameid, date, champion, teamname
    FROM '{CSV}'
    WHERE league = 'LCK' AND position != 'team'
''').df()

oe_games = {}
unmapped = set()
for gameid, grp in oe.groupby("gameid"):
    ids = set()
    for n in grp["champion"]:
        cid = name_to_id.get(n)
        if cid is None: unmapped.add(n)
        else: ids.add(cid)
    oe_games[gameid] = {
        "date": str(grp["date"].iloc[0])[:10],
        "champs": frozenset(ids),
        "teams": frozenset(grp["teamname"].unique()),
    }
print("unmapped champion names:", unmapped or "none — OE and Data Dragon agree on all spellings")


unmapped champion names: none — OE and Data Dragon agree on all spellings


In [7]:
# ── Match LP↔OE on (date ±1, champion set); verify team names ────────────
from datetime import date, timedelta

lp_index = {}
for g in cache.values():
    champ_set = frozenset(pb.championId for pb in g.picksBans if not pb.isBan)
    lp_index[(g.start[:10], champ_set)] = g

def close_dates(d):
    base = date.fromisoformat(d)
    return [str(base + timedelta(days=k)) for k in (-1, 0, 1)]

matched, name_mismatches = [], []
for gameid, og in oe_games.items():
    hit = next((lp_index[(d, og["champs"])] for d in close_dates(og["date"])
                if (d, og["champs"]) in lp_index), None)
    if hit is None:
        continue                    # expected for most: cache only holds 15 games
    matched.append(gameid)
    lp_teams = frozenset([hit.teams.BLUE.sources.leaguepedia.name,
                          hit.teams.RED.sources.leaguepedia.name])
    if lp_teams != og["teams"]:
        name_mismatches.append((gameid, og["teams"], lp_teams))

print(f"LP games matched to OE: {len(matched)} / {len(cache)}")
print("team-name mismatches:", name_mismatches or "none — names identical across sources")


LP games matched to OE: 15 / 15
team-name mismatches: [('LOLTMNT02_241605', frozenset({'Kiwoom DRX', 'DN SOOPers'}), frozenset({'DRX', 'DN Freecs'})), ('LOLTMNT02_241948', frozenset({'HANJIN BRION', 'Nongshim RedForce'}), frozenset({'Nongshim RedForce', 'OKSavingsBank BRION'})), ('LOLTMNT02_241955', frozenset({'HANJIN BRION', 'Nongshim RedForce'}), frozenset({'Nongshim RedForce', 'OKSavingsBank BRION'})), ('LOLTMNT02_242586', frozenset({'Kiwoom DRX', 'DN SOOPers'}), frozenset({'DRX', 'DN Freecs'})), ('LOLTMNT02_243267', frozenset({'Kiwoom DRX', 'T1'}), frozenset({'T1', 'DRX'})), ('LOLTMNT02_244166', frozenset({'Kiwoom DRX', 'T1'}), frozenset({'T1', 'DRX'})), ('LOLTMNT02_244356', frozenset({'HANJIN BRION', 'BNK FEARX'}), frozenset({'BNK FEARX', 'OKSavingsBank BRION'}))]


## Decision note (0.2b) — fill in after the match run

- **Match rate:** 15 / 15 via (date ±1, champion-set) key.
- **Unmapped champion names:** none — no alias table needed.
- **Team-name mismatches:** team-name mismatches: [('LOLTMNT02_241605', frozenset({'Kiwoom DRX', 'DN SOOPers'}), frozenset({'DRX', 'DN Freecs'})), ('LOLTMNT02_241948', frozenset({'HANJIN BRION', 'Nongshim RedForce'}), frozenset({'Nongshim RedForce', 'OKSavingsBank BRION'})), ('LOLTMNT02_241955', frozenset({'HANJIN BRION', 'Nongshim RedForce'}), frozenset({'Nongshim RedForce', 'OKSavingsBank BRION'})), ('LOLTMNT02_242586', frozenset({'Kiwoom DRX', 'DN SOOPers'}), frozenset({'DRX', 'DN Freecs'})), ('LOLTMNT02_243267', frozenset({'Kiwoom DRX', 'T1'}), frozenset({'T1', 'DRX'})), ('LOLTMNT02_244166', frozenset({'Kiwoom DRX', 'T1'}), frozenset({'T1', 'DRX'})), ('LOLTMNT02_244356', frozenset({'HANJIN BRION', 'BNK FEARX'}), frozenset({'BNK FEARX', 'OKSavingsBank BRION'}))] - luckily we did not join via team names, since they change the sponsor and so the name...
  
- **Costs measured:** bot-password auth required; ~10–30s effective per detail call (2 parallel requests each); connection drops under load; morning fetches much faster than night.
- **Verdict:** OE = backbone incl. full draft order via verified weave rules; Leaguepedia = supplement — `gameInSeries` from cheap skeleton tier, detail calls only where player metadata matters (Phases 4/7), always via cached background scripts; joins on champion-set, never on names; name variants handled by an alias table seeded from spike output.